# Anomaly analysis — TR-004 fuel turbopump (synthetic LUMEN-inspired showcase)

**This notebook works against the seeded showcase Collection** (`examples/lumen-showcase/seed.py`).
All data is synthetic and only loosely inspired by DLR's LUMEN demonstrator — see
[`docs/showcase.md`](../../docs/showcase.md) and the seed README for the full disclaimer.

What this notebook demonstrates:

1. Connect via the shepard Python client and a reviewer-role API key.
2. Walk the campaign Collection → list its DataObjects → fetch each test run's `vib_fuel_pump` channel.
3. Plot all seven runs on a 7-panel grid; the TR-004 spike is unmistakable.
4. Apply a rolling-median ± k·MAD detector and report the spike location.
5. Show TR-004's lab-journal debrief alongside the plot.
6. Confirm TR-006's `vib_fuel_pump` is back inside the nominal envelope.
7. Build the **proposed R2c-shape** `ExportSelection` JSON for a redacted bundle.
8. Best-effort post the export and inspect the `ro-crate-metadata.json` selection block.
9. Wrap up.

Optional dependencies: `pandas`, `matplotlib`, `jupyter`. The notebook prints a friendly hint and continues if they are missing.

In [ ]:
# 1. Connect.
import os
import json
from pathlib import Path

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    print('hint: pip install matplotlib for the plots; continuing without them')
    HAVE_MPL = False

try:
    import pandas as pd
    HAVE_PD = True
except ImportError:
    print('hint: pip install pandas for tabular display; continuing without it')
    HAVE_PD = False

from shepard_client import (
    ApiClient, Configuration,
    CollectionApi, CollectionSearchBody, CollectionSearchParams,
    DataObjectApi,
    SearchApi,
    TimeseriesContainerApi, TimeseriesReferenceApi,
    LabJournalEntryApi,
    SemanticAnnotationApi,
)

HOST = os.environ.get('BACKEND_URL', 'http://localhost:8080/shepard/api')
APIKEY = os.environ.get('REVIEWER_API_KEY') or os.environ.get('API_KEY')
if not APIKEY:
    raise SystemExit('set REVIEWER_API_KEY (or API_KEY) in the env first')

client = ApiClient(Configuration(host=HOST, api_key={'apikey': APIKEY}))
coll_api = CollectionApi(client)
do_api = DataObjectApi(client)
search_api = SearchApi(client)
tsc_api = TimeseriesContainerApi(client)
tsr_api = TimeseriesReferenceApi(client)
journal_api = LabJournalEntryApi(client)
ann_api = SemanticAnnotationApi(client)
print(f'connected to {HOST}')

In [ ]:
# 2. Walk the campaign Collection.
COLLECTION_NAME = 'LUMEN-Inspired Hotfire Test Campaign — Q3 2024 (synthetic)'
search = search_api.search_collections(
    CollectionSearchBody(searchParams=CollectionSearchParams(
        query=f'{{"property":"name","value":"{COLLECTION_NAME}","operator":"eq"}}'
    ))
)
if not search.results:
    raise SystemExit(f'no Collection {COLLECTION_NAME!r} — run seed.py first')
coll = search.results[0]
print(f'collection id={coll.id}  name={coll.name!r}')

all_dos = do_api.get_all_data_objects(coll.id) or []
runs = sorted([d for d in all_dos if d.name.startswith('TR-')], key=lambda d: d.name)
investigation = next((d for d in all_dos if d.name.startswith('Anomaly Investigation')), None)
for d in runs:
    print(f'  {d.name}  id={d.id}  fired={d.attributes.get("is_fired", "?")}')
if investigation is not None:
    print(f'  ! {investigation.name}  id={investigation.id}  closed_at={investigation.attributes.get("closed_at", "")!r}')

In [ ]:
# Fetch the vib_fuel_pump channel for every fired run.
def vib_for_run(run_do):
    # operationId: getAllTimeseriesReferences
    refs = tsr_api.get_all_timeseries_references(coll.id, run_do.id) or []
    if not refs:
        return None, None
    tsr = refs[0]
    points = []
    for ts in tsr.timeseries or []:
        if ts.symbolic_name == 'vib_fuel_pump':
            # OperationId getTimeseries on TimeseriesContainerApi
            # (path: /timeseriesContainers/{id}/payload).
            payload = tsc_api.get_timeseries(
                timeseries_container_id=tsr.timeseries_container_id,
                start=tsr.start,
                end=tsr.end,
                measurement=ts.measurement,
                device=ts.device,
                location=ts.location,
                symbolic_name=ts.symbolic_name,
                field=ts.field,
            )
            # The payload is a TimeseriesWithDataPoints; extract `points`.
            for p in (payload.points or []):
                points.append((p.timestamp, p.value))
            break
    if not points:
        return None, None
    points.sort()
    ts_ns = np.array([p[0] for p in points], dtype=np.int64)
    val = np.array([p[1] for p in points], dtype=np.float64)
    t_s = (ts_ns - ts_ns[0]) / 1e9
    return t_s, val

vib_traces = {}
for r in runs:
    t, v = vib_for_run(r)
    if t is None:
        print(f'{r.name}: no vib_fuel_pump (probably hold day)')
        continue
    vib_traces[r.name] = (t, v)
    print(f'{r.name}: {len(t)} samples, peak {v.max():.2f} g rms at t={t[v.argmax()]:.2f} s')

In [ ]:
# 3. Plot all 7 runs on a 7-panel grid.
if HAVE_MPL and vib_traces:
    rows = sorted(vib_traces)
    fig, axes = plt.subplots(len(rows), 1, figsize=(9, 1.8 * len(rows)), sharex=True)
    for ax, name in zip(np.atleast_1d(axes), rows):
        t, v = vib_traces[name]
        ax.plot(t, v, lw=0.6)
        ax.axhline(4.0, color='red', ls='--', lw=0.5, label='4 g rms envelope')
        ax.set_ylabel(name)
        ax.set_ylim(0, 14)
    axes[-1].set_xlabel('seconds since ignition')
    fig.suptitle('vib_fuel_pump across the campaign (synthetic)')
    fig.tight_layout()
    plt.show()

In [ ]:
# 4. Rolling-median ± k·MAD detector.
def rolling_mad_detect(t, v, window=51, k=6.0):
    if window % 2 == 0:
        window += 1
    half = window // 2
    pad_v = np.pad(v, (half, half), mode='edge')
    med = np.array([np.median(pad_v[i:i + window]) for i in range(len(v))])
    mad = np.array([np.median(np.abs(pad_v[i:i + window] - med[i])) for i in range(len(v))])
    mad = np.maximum(mad, 1e-3)  # floor so we don't divide by zero in quiet stretches
    z = (v - med) / (1.4826 * mad)
    return z, np.abs(z) > k

for name, (t, v) in vib_traces.items():
    z, mask = rolling_mad_detect(t, v, window=51, k=6.0)
    if mask.any():
        i0, i1 = np.argmax(mask), len(mask) - 1 - np.argmax(mask[::-1])
        peak = v.max()
        print(f'{name}: anomaly window t={t[i0]:.2f}..{t[i1]:.2f} s  peak={peak:.2f} g rms')
    else:
        print(f'{name}: clean')


In [ ]:
# 5. Pull TR-004's lab-journal entry and display alongside the plot.
tr4 = next(r for r in runs if r.name == 'TR-004')
# operationId: getLabJournalsByCollection (filtered by data_object_id)
entries = journal_api.get_lab_journals_by_collection(data_object_id=tr4.id) or []
for e in entries:
    print('---')
    print(e.journal_content)
if HAVE_MPL and 'TR-004' in vib_traces:
    t, v = vib_traces['TR-004']
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(t, v, lw=0.7)
    ax.axhline(4.0, color='red', ls='--', lw=0.6)
    ax.axvspan(7.65, 8.45, color='orange', alpha=0.25, label='detected anomaly')
    ax.set_xlabel('seconds')
    ax.set_ylabel('TR-004 vib_fuel_pump (g rms)')
    ax.legend()
    plt.show()

In [ ]:
# 6. Confirm TR-006 is back inside the envelope.
if 'TR-006' in vib_traces:
    t, v = vib_traces['TR-006']
    print(f'TR-006 vib_fuel_pump: peak {v.max():.2f} g rms, mean {v.mean():.2f} g rms')
    print('  expected: peak ~3.6 g rms (ramp_up); steady_state ≤ 2.4 g rms')
else:
    print('TR-006 not seeded — re-run seed.py')

In [ ]:
# 7. Build the R2c-shape selective-export request body.
# This is the proposed shape of the future ExportSelection body. The current
# /collections/{id}/export endpoint is GET-only; we build the body anyway so
# the showcase doc has an exact reference, and so the notebook is ready for
# the day R2c ships.
if investigation is None:
    raise SystemExit('investigation DataObject not seeded — re-run seed.py')

selection = {
    'collection_id': coll.id,
    'roots': [tr4.id, investigation.id],
    'include': {
        'descendants': True,
        'predecessors': True,
        'references': ['timeseries', 'files', 'structured'],
        'annotations': True,
        'lab_journal_entries': True,
    },
    'redact': ['LAB_JOURNAL_CONTENT'],
    'format': 'rocrate-zip',
}
print(json.dumps(selection, indent=2))

In [ ]:
# 8. Best-effort: POST the export, fall back to GET on 404/405.
import urllib.error
import urllib.request
import zipfile
import io

url = f'{HOST.rstrip("/")}/collections/{coll.id}/export'
headers = {'apikey': APIKEY}
post_body = json.dumps(selection).encode('utf-8')
post_headers = dict(headers)
post_headers['Content-Type'] = 'application/json'
zip_bytes = None
try:
    req = urllib.request.Request(url, data=post_body, headers=post_headers, method='POST')
    with urllib.request.urlopen(req, timeout=60) as resp:
        zip_bytes = resp.read()
        print(f'POST export: HTTP {resp.status}, {len(zip_bytes)} bytes')
except urllib.error.HTTPError as e:
    if e.code in (404, 405, 415, 501):
        print(f'POST export rejected ({e.code}); falling back to GET (R2c not yet shipped)')
        req = urllib.request.Request(url, headers=headers, method='GET')
        with urllib.request.urlopen(req, timeout=60) as resp:
            zip_bytes = resp.read()
            print(f'GET export: HTTP {resp.status}, {len(zip_bytes)} bytes')
    else:
        print(f'POST export failed: HTTP {e.code} {e.reason}')
        raise

if zip_bytes is not None:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        for name in zf.namelist()[:20]:
            print('  ', name)
        meta_name = next((n for n in zf.namelist() if n.endswith('ro-crate-metadata.json')), None)
        if meta_name:
            meta = json.loads(zf.read(meta_name))
            graph_kinds = {item.get('@type', '?') for item in meta.get('@graph', [])}
            print('manifest @types:', sorted(graph_kinds))
            # Foreshadowing R2c: the future selection block will live under
            # an explicit `selection` key on the root entity. We print our
            # local selection record for parity.
            print('local selection record:')
            print(json.dumps(selection, indent=2))

## 9. What we just demonstrated

- Walked the seven-run campaign Collection via the shepard Python client.
- Pulled the `vib_fuel_pump` channel for each run, plotted them on a 7-panel grid, and saw the TR-004 spike.
- A rolling-median ± k·MAD detector flagged exactly the TR-004 spike and reported its window.
- Read TR-004's debrief lab-journal entry and rendered the plot beside it.
- Verified TR-006 (re-fire after bearing replacement) is back inside the nominal envelope.
- Built the proposed R2c `ExportSelection` JSON, attempted a selective POST export, and fell back gracefully when the endpoint rejected the body (R2c is not yet shipped).

For the wider feature tour see [`docs/showcase.md`](../../docs/showcase.md).